# Lab 1.1 — Python + ML Warmup with Traffic Flow Data

In this lab you will:

1. Generate a synthetic traffic-flow dataset (speed and density).
2. Visualize the **fundamental diagram** — the relationship between speed and density.
3. Fit a simple supervised-learning model and evaluate it.

The point of this lab is light: confirm your Python environment works, refresh your memory of the supervised-learning pipeline (data → model → loss → evaluation), and connect a familiar transportation-engineering concept (the fundamental diagram) to the AI workflow.

If you're running this in **Google Colab**, no setup is needed — just run the cells. If you're running locally, make sure you've installed `numpy`, `pandas`, `matplotlib`, and `scikit-learn`.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(seed=42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

## 2. Generate synthetic traffic-flow data

We'll generate data from the **Greenshields linear speed–density model**:

$$ v(k) = v_f \left(1 - \frac{k}{k_j}\right) $$

where:
- $v$ = mean traffic speed (mph)
- $k$ = density (veh/mile/lane)
- $v_f$ = free-flow speed = 65 mph
- $k_j$ = jam density = 200 veh/mile/lane

We add Gaussian noise to simulate the variability you'd see in real loop-detector data.


In [ ]:
# True parameters
v_f = 65.0   # free-flow speed (mph)
k_j = 200.0  # jam density (veh/mi/lane)

# Sample densities and compute noisy speeds
n = 250
k = rng.uniform(0, k_j, size=n)
v_true = v_f * (1 - k / k_j)
noise = rng.normal(0, 4.0, size=n)   # 4 mph std measurement noise
v = np.clip(v_true + noise, 0, None)

df = pd.DataFrame({'density_veh_per_mile': k, 'speed_mph': v})
df.head()

## 3. Visualize the fundamental diagram

Plot speed against density. You should see the familiar downward-sloping relationship — higher density, lower mean speed.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['density_veh_per_mile'], df['speed_mph'], s=18, alpha=0.7)
ax.set_xlabel('Density (veh/mile/lane)')
ax.set_ylabel('Mean speed (mph)')
ax.set_title('Synthetic speed–density fundamental diagram')
plt.show()

## 4. Fit a simple supervised-learning model

We treat **density** as the input feature and **speed** as the target. The Greenshields model is linear in density, so plain linear regression should fit well — though real-world data is usually more nonlinear (Greenberg, Underwood, and other models exist for a reason).

We'll do a 70/30 train/test split so we can evaluate the model on held-out data.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['density_veh_per_mile']].values
y = df['speed_mph'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Estimated free-flow speed v_f ≈ {model.intercept_:.2f} mph')
print(f'Estimated -v_f / k_j slope    ≈ {model.coef_[0]:.4f}')
print(f'Implied jam density k_j       ≈ {-model.intercept_ / model.coef_[0]:.1f} veh/mile')

## 5. Evaluate


In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'Test RMSE: {rmse:.2f} mph')
print(f'Test R^2 : {r2:.3f}')

# Plot fit on top of the data
k_grid = np.linspace(0, k_j, 200).reshape(-1, 1)
v_grid = model.predict(k_grid)

fig, ax = plt.subplots()
ax.scatter(X[:, 0], y, s=18, alpha=0.5, label='Observations')
ax.plot(k_grid, v_grid, linewidth=2, label='Linear-regression fit')
ax.set_xlabel('Density (veh/mile/lane)')
ax.set_ylabel('Mean speed (mph)')
ax.set_title('Greenshields model recovered from noisy data')
ax.legend()
plt.show()

## 6. Discussion

Things to notice:

- The regression recovered the underlying free-flow speed and jam density — even with 4 mph noise per observation.
- The data-generating process was *exactly* linear here. Real loop-detector data is not linear, and the choice of fundamental-diagram functional form (Greenshields, Greenberg, Underwood, Edie, Pipes-Munjal, …) matters when you're calibrating real-world models.
- **This is the entire supervised-learning pipeline in miniature.** Generate or collect labeled data, split into train/test, fit a model, evaluate on held-out points. Everything we do in Modules 2–4 — vehicle detection, lane-keeping, signal control — is variations on this skeleton, with bigger models and richer inputs.

## 7. Exercises

1. **Add a flow column.** Flow is $q = k \cdot v$ in vehicles/hour/lane. Compute it and plot the **flow–density fundamental diagram**. Notice that flow vs density is parabolic under the Greenshields assumption — linear regression won't fit it directly. What feature transformation would let a linear model capture it?

2. **Compare functional forms.** Refit the speed–density relationship with a polynomial regression of degree 2 and degree 3. Does test-set RMSE improve, or does the higher-order model just overfit the noise?

3. **Real data.** Download a short window of PeMS or NGSIM data, compute aggregated speed and density, and try the same fit. What looks different, and why?

Share solutions or questions via the repository's issue tracker.